# Ingest and transform Victorian weather forecasts

One daily snapshot at noon AEST, covering the next six days hourly.

Schema: `forecast_origin | time | location | temperature | humidity`.

Nine locations: seven terminal proxies, Melbourne Olympic Park and Melbourne Airport. Coordinates approximate demand locations.

## Pipeline structure

One daily origin → nine locations → 144 hours per location → Silver Delta table.

## Libraries

`requests`: API calls. `pandas`: response tables. Spark: Delta writes.

In [ ]:
%pip install requests pandas

## Configuration

Model: ECMWF IFS HRES 9 km. Archive starts 14 March 2024; noon snapshots start 15 March.

Times use fixed AEST (UTC+10). `TIMESTAMP_NTZ` preserves this clock without daylight saving conversion.

In [ ]:
import pandas as pd
import requests

API_URL = "https://single-runs-api.open-meteo.com/v1/forecast"
TABLE_NAME = "workspace.default.weather_forecast_vic_hourly"
START_DATE = "2024-03-15"
FORECAST_DAYS = 6
ORIGIN_HOUR = 12
MODEL = "ecmwf_ifs"

LOCATIONS = [
    {"name": "Keilor", "code": "keilor", "latitude": -37.7382388900787, "longitude": 144.8461749995},
    {"name": "Rowville", "code": "rowville", "latitude": -37.9292805597441, "longitude": 145.227283300257},
    {"name": "Cranbourne", "code": "cranbourne", "latitude": -38.0856333301827, "longitude": 145.266433299488},
    {"name": "Richmond", "code": "richmond", "latitude": -37.8311361095565, "longitude": 145.001580600346},
    {"name": "Thomastown", "code": "thomastown", "latitude": -37.6936999995828, "longitude": 145.009927799778},
    {"name": "East Rowville", "code": "east_rowville", "latitude": -37.940958329999, "longitude": 145.236341700332},
    {"name": "Geelong", "code": "geelong", "latitude": -38.0826249998139, "longitude": 144.338497200197},
    {"name": "Melbourne Olympic Park", "code": "melbourne_olympic_park", "latitude": -37.8255, "longitude": 144.9816},
    {"name": "Melbourne Airport", "code": "melbourne_airport", "latitude": -37.6655, "longitude": 144.8321},
]

now = pd.Timestamp.now(tz="Australia/Brisbane").tz_localize(None)
origins = pd.date_range(START_DATE, now.normalize(), freq="D") + pd.Timedelta(hours=ORIGIN_HOUR)
origins = origins[origins <= now]

## Daily origins and resolution

`forecast_origin`: noon AEST snapshot. `time`: predicted weather hour.

Each snapshot uses the previous day’s 12:00 UTC run, initialized 14 hours earlier. This leaves a publication buffer.

| After model initialization | After our noon snapshot | Native spacing | Hourly output |
| --- | --- | --- | --- |
| Through 90 hours | Through 76 hours | 1 hour | Native steps |
| After 90 through 144 hours | After 76 through 130 hours | 3 hours | Intermediate hours interpolated |
| After 144 hours | After 130 through 144 hours of our window | 6 hours | Intermediate hours interpolated |

Interpolation fills gaps between model steps. [Resolution source](https://open-meteo.com/en/docs/ecmwf-api).

Older archive entries are reconstructed IFS Cycle 49R1 hindcasts, not verified historical publication snapshots. [Archive source](https://open-meteo.com/en/docs/single-runs-api).

In [ ]:
COLUMNS = ["forecast_origin", "time", "location", "temperature", "humidity"]

PARAMS = {
    "latitude": ",".join(str(location["latitude"]) for location in LOCATIONS),
    "longitude": ",".join(str(location["longitude"]) for location in LOCATIONS),
    "hourly": "temperature_2m,relative_humidity_2m",
    "models": MODEL,
    "forecast_days": FORECAST_DAYS + 2,
    "timezone": "Australia/Brisbane",
}

def ingest_weather(origin):
    origin = pd.Timestamp(origin)
    model_run = origin.tz_localize("Australia/Brisbane").tz_convert("UTC") - pd.Timedelta(hours=14)
    params = {**PARAMS, "run": model_run.strftime("%Y-%m-%dT%H:%M")}
    response = requests.get(API_URL, params=params, timeout=60)
    response.raise_for_status()
    output = response.json()
    assert isinstance(output, list) and len(output) == len(LOCATIONS), "Incomplete location response"

    frames = []
    expected_times = pd.date_range(origin + pd.Timedelta(hours=1), periods=FORECAST_DAYS * 24, freq="h")
    for location, data in zip(LOCATIONS, output):
        frame = pd.DataFrame(data["hourly"]).rename(columns={
            "temperature_2m": "temperature", "relative_humidity_2m": "humidity",
        })
        frame["time"] = pd.to_datetime(frame["time"])
        frame = frame[frame["time"].isin(expected_times)].copy()
        assert pd.DatetimeIndex(frame["time"]).equals(expected_times), f"Incomplete hours: {origin}, {location['name']}"
        assert frame[["temperature", "humidity"]].notna().all().all(), f"Missing weather: {origin}, {location['name']}"
        frame["forecast_origin"] = origin
        frame["location"] = location["name"]
        frames.append(frame[COLUMNS])

    weather_long = pd.concat(frames, ignore_index=True)
    weather_spark = spark.createDataFrame(weather_long,
        "forecast_origin timestamp_ntz, time timestamp_ntz, location string, temperature double, humidity double")
    weather_spark.write.format("delta").mode("append").saveAsTable(TABLE_NAME)
    print(f"Saved {origin}: {len(weather_long)} rows")

## Retrieve and save one origin

`ingest_weather(origin)` retrieves and appends one snapshot. Append creates the table if absent.

Run the historical loop once. For daily updates, call `ingest_weather(origin)` with that day’s noon AEST timestamp. Repeating an origin appends duplicates.

In [ ]:
for origin in origins:
    ingest_weather(origin)

## Inspect the saved table

Show origin and target ranges, location counts and nulls. Daily execution must be scheduled separately.

In [ ]:
data = spark.table(TABLE_NAME)
data.selectExpr(
    "count(*) AS rows", "count(distinct forecast_origin) AS origins",
    "min(forecast_origin) AS first_origin", "max(forecast_origin) AS last_origin",
    "min(time) AS first_time", "max(time) AS last_time",
    "count(distinct location) AS locations",
    *[f"count_if({column} IS NULL) AS null_{column}" for column in COLUMNS],
).show(truncate=False)
data.orderBy("forecast_origin", "location", "time").show(5, truncate=False)